# 07. UGV 카메라 영상 확인 (Camera)

영상 대응: **26:51–28:14**

> 3차 교육 영상 20–30분의 기능을 재구성한 **로컬 실습본**입니다. 공식 원본 노트북이 아닙니다. `mil_msgs` 원본 정의가 없어 이 환경에서는 `sim_mil_interfaces`를 사용합니다. 실제 ROS 2로 시뮬레이터와 통신합니다.

`/image_raw`의 `sensor_msgs/msg/Image`를 구독합니다. `cv_bridge`로 OpenCV 배열을 만든 뒤 JPEG로 변환하여 JupyterLab에 표시합니다.

| 항목 | 영상 설명 |
|---|---|
| 해상도 | 640 × 480 |
| 영상 형식 | BGR, OpenCV |
| 카메라 최대 프레임 | 공식 장치 최대 30 FPS, CPU 부하를 고려해 10 FPS 이하 권장 |
| 예제 출력 | 사진 한 장. 실시간 스트리밍 예제가 아님 |

현재 시뮬레이터의 실제 프레임 주기는 Mac·VM 부하에 따라 달라집니다. 센서가 없거나 영상이 늦으면 오류를 표시합니다. 깊이 영상 API는 이 실습에 포함하지 않습니다.


## 실행 방법

위쪽 커널을 **Python 3 (ROS 2 · simulation)** 으로 선택하고, 아래 셀을 위에서부터 **Shift+Enter**로 실행합니다. 물리 경기장과 ROS 실습 연결을 먼저 켜세요. 각 실습은 시간이 지나면 종료되며, 셀을 다시 실행할 수 있습니다.


In [ ]:
import time
import math
import uuid
import rclpy
from rclpy.qos import QoSProfile, ReliabilityPolicy, DurabilityPolicy, qos_profile_sensor_data
from rclpy.signals import SignalHandlerOptions
from sim_mil_interfaces.msg import State as MissionState, Match as MatchCommand

# 상태/경기 토픽은 마지막 값을 새 구독자에게도 전달합니다.
STATUS_QOS = QoSProfile(depth=1, reliability=ReliabilityPolicy.RELIABLE,
                        durability=DurabilityPolicy.TRANSIENT_LOCAL)

def new_node(prefix):
    if not rclpy.ok():
        # Jupyter의 중단 버튼이 KeyboardInterrupt와 finally 정지를 처리하게 둡니다.
        rclpy.init(args=[], signal_handler_options=SignalHandlerOptions.NO)
    return rclpy.create_node(prefix + '_' + uuid.uuid4().hex[:8])

def spin_until(node, predicate, timeout=5.0):
    deadline = time.monotonic() + timeout
    while not predicate() and time.monotonic() < deadline:
        rclpy.spin_once(node, timeout_sec=min(0.05, max(0.0, deadline-time.monotonic())))
    if not predicate():
        raise TimeoutError('ROS 데이터/응답 시간 초과. 물리 경기장과 ROS 연결 상태를 확인하세요.')

def watch_status(node):
    data = {'state': None, 'match': None, 'state_at': None}
    def on_state(msg):
        data['state'] = msg
        data['state_at'] = time.monotonic()
    def on_match(msg):
        data['match'] = msg
    subscriptions = [
        node.create_subscription(MissionState, '/mil/state', on_state, STATUS_QOS),
        node.create_subscription(MatchCommand, '/mil/match', on_match, STATUS_QOS),
    ]
    spin_until(node, lambda: data['state'] is not None and data['match'] is not None)
    return data, subscriptions

def allowed(data):
    if data['state_at'] is None or time.monotonic()-data['state_at'] > 1.5:
        return False
    match = data['match']
    if match is None or match.cmd == 'pause':
        return False
    return match.mode == 'practice' or (match.mode == 'mission' and match.cmd in ('start', 'resume'))

def require_control(data):
    if not allowed(data):
        raise RuntimeError('제어 잠금 또는 상태 데이터 지연. 경기 재개·연결 상태를 확인하세요.')

def wait_publisher(node, publisher, timeout=5.0):
    spin_until(node, lambda: publisher.get_subscription_count() > 0, timeout)

def yaw_degrees(q):
    return math.degrees(math.atan2(2.0*(q.w*q.z + q.x*q.y), 1.0-2.0*(q.y*q.y+q.z*q.z)))

def source_age_seconds(node, message, maximum, label):
    stamp = message.header.stamp
    source_ns = int(stamp.sec)*1_000_000_000 + int(stamp.nanosec)
    if source_ns <= 0:
        raise ValueError(label + ': 원본 센서 시각이 없습니다.')
    age = (node.get_clock().now().nanoseconds-source_ns)/1_000_000_000
    if age < -0.25:
        raise ValueError(f'{label}: 센서 시각이 {-age:.3f}초 미래입니다. ROS 시계 설정을 확인하세요.')
    if age > maximum:
        raise TimeoutError(f'{label}: 원본 데이터가 {age:.3f}초 전 값입니다.')
    return age


## 실습 설정

아래 값을 바꾸고 다시 실행할 수 있습니다.


In [ ]:
TIMEOUT_SECONDS = 10.0
MAX_FRAME_AGE_SECONDS = 2.0


## 실행


In [ ]:
import cv2
from cv_bridge import CvBridge
from sensor_msgs.msg import Image as RosImage
from IPython.display import display, Image as NotebookImage
node = new_node('demo_camera')
latest = {'image':None, 'at':None}
def on_image(msg):
    latest['image'], latest['at'] = msg, time.monotonic()
try:
    subscription = node.create_subscription(RosImage, '/image_raw', on_image, qos_profile_sensor_data)
    spin_until(node, lambda: latest['image'] is not None, TIMEOUT_SECONDS)
    msg = latest['image']
    age = time.monotonic()-latest['at']
    if age > MAX_FRAME_AGE_SECONDS:
        raise TimeoutError('카메라 수신 데이터가 오래됐습니다.')
    source_age = source_age_seconds(node, msg, MAX_FRAME_AGE_SECONDS, 'Camera')
    if not msg.data or msg.width <= 0 or msg.height <= 0:
        raise ValueError('비어 있거나 잘못된 카메라 프레임입니다.')
    frame = CvBridge().imgmsg_to_cv2(msg, desired_encoding='bgr8')
    ok, jpg = cv2.imencode('.jpg', frame)
    if not ok:
        raise RuntimeError('카메라 JPEG 변환 실패')
    print('Camera:', frame.shape, '| source:', msg.encoding)
    display(NotebookImage(data=jpg.tobytes(), width=640))
    DEMO_RESULT = {'demo':'07','shape':list(frame.shape),'encoding':msg.encoding,'bytes':len(jpg),
                   'received_age_seconds':age,'source_age_seconds':source_age,'frame_id':msg.header.frame_id,'source_stamp':{'sec':msg.header.stamp.sec,'nanosec':msg.header.stamp.nanosec}}
finally:
    node.destroy_node()


[← 실습 목차](00.실습목차.ipynb) · [코드 편집](../team_node.py)

실물 장치의 토픽 타입·주기·응답과 맞는지는 지급 UGV에서 확인해야 합니다.
